# Хронологія подій головного героя (Subject - Verb - Object)

In [1]:
import json
import os

import pandas as pd
from dotenv import load_dotenv
from google import genai
from google.genai import types


In [2]:
INPUT_FILE = "../lab01/Zabrodin_file_2.json"
OUTPUT_FILE = "Zabrodin_subject_verb_object.csv"
CHARACTER = "Мотря"

with open(INPUT_FILE, 'r', encoding='utf-8') as f:
    data = json.load(f)
    conllu_text = data['result']


## Парсинг JSON відповіді від UDPipe

In [3]:
def parse(conllu_data: str) -> list[dict]:
    sentences = []
    current = {}

    for line in conllu_data.splitlines():
        if line.startswith("#") or not line:
            if not line and current:
                sentences.append(current)
                current = {}
            continue

        fields = line.split("\t")
        if len(fields) < 10 or "-" in fields[0]:
            continue

        current[fields[0]] = {
            'id': fields[0],
            'form': fields[1],
            'lemma': fields[2],
            'pos': fields[3],
            'head': fields[6],
            'deprel': fields[7],
        }

    if current:
        sentences.append(current)

    return sentences

## Пошук трійкок слів

In [4]:
def process(words: dict, character: str) -> list[dict]:
    rows = []

    for word_info in words.values():
        # Лаврін СМІЯВСЯ
        # Лаврін був ПОБИТИЙ
        if word_info['deprel'] not in ('nsubj', 'nsubj:pass') or word_info['lemma'].lower() != character.lower():
            continue

        verb_id = word_info['head']
        if verb_id not in words:
            continue

        verbs = [words[verb_id]] + [
            w for w in words.values()
            if w['head'] == verb_id and w['deprel'] == 'conj' and w['pos'] == 'VERB'
        ]

        for verb in verbs:
            # Лаврін почав СМІЯТИСЯ
            # Лаврін став ПРАЦЮВАТИ
            xcomp = next(
                (w for w in words.values() if w['head'] == verb['id'] and w['deprel'] == 'xcomp'),
                None
            )

            # Лаврін стругав ВІСЬ
            # Лаврін пішов ДОДОМУ
            # Лаврін дав КАРПУ м'яч
            obj = next(
                (w for w in words.values() if w['head'] == verb['id'] and w['deprel'] in ('obj', 'obl', 'iobj')),
                None
            )

            rows.append({
                'Subject': word_info['form'],
                'Verb': f"{verb['form']} {xcomp['form']}" if xcomp else verb['form'],
                'Object': obj['form'] if obj else "-",
            })

    return rows

In [5]:
def extract_subject_verb_object(conllu_data: str, character: str) -> pd.DataFrame:
    rows = []
    for sentence in parse(conllu_data):
        rows.extend(process(sentence, character))
    return pd.DataFrame(rows)

In [6]:
df = extract_subject_verb_object(conllu_text, CHARACTER)
df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")
display(df.head(10))

,Subject,Verb,Object
0,Мотря,виходила,його
1,Мотря,дивилась,його
2,Мотря,жила,-
3,Мотря,вийшла,хати
4,Мотря,обізвалась,-
5,Мотря,стояла,хатою
6,Мотря,сказала,-
7,Мотря,подала,руки
8,Мотря,підставила,глиняника
9,Мотря,крикнула,двір


## Відновлення хронології за допомогою Gemini API

In [7]:
load_dotenv()

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))


def restore_chronology(df: pd.DataFrame, character: str) -> str:
    uploaded_file = client.files.upload(
        file=OUTPUT_FILE,
        config={"mime_type": "text/csv"}
    )

    prompt = (
        f"Відповідай лише звичайним текстом, без Markdown форматування.\n"
        f"Відповідь має бути українською мовою.\n"
        f"Ось список дій персонажа {character} у хронологічному порядку у формі [Subject],[Verb],[Object].\n"
        f"Якщо у останньому стовпчику знак '-' - це означає, що не було знайдено значення.\n"
        f"Спробуй відновити хронологію дій персонажа у вигляді зв'язного тексту.\n"
        f"Текст має бути логічним та приємним для читання людиною."
    )

    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=[uploaded_file, prompt],
        config=types.GenerateContentConfig(
            thinking_config=types.ThinkingConfig(thinking_budget=1024)
        )
    )

    return response.text


chronology = restore_chronology(df, CHARACTER)
print(chronology)

Мотря постійно працювала по господарству. Вона виходила до Карпа, дивилася на нього, виходила з хати, обзивалася та подавала руки. Коли потрібно було, вона кричала на двір, підставляла глиняника, інколи задріботіла або блиснула рядками зубів. Вона працювала не покладаючи рук: прибирала у неділю, вбиралася в ошатну спідницю, підперезувалася поясом, взувала чоботи та йшла до церкви чи на танець.

Життя в сім’ї було непростим. Мотря поралася в хаті: видоїла корову, процідила молоко, натопила печі та приготувала обід. Вона сиділа за столом, спускала очі, здержувала язика, намагаючись не сперечатися зі свекрухою. Проте конфлікти були неминучими. Коли терпець уривався, Мотря кидала лаву, виходила з хати, сварилася, кричала, а іноді навіть хапалася за хворостину чи деркач.

Часто виникали гострі суперечки. Мотря вибігала з хати, репетувала, хапала Мелашку або замахнулася на когось. Вона бігала до волості, лазила на горище, збирала гнізда в курнику, але спокою не було ніде. Вона часто кричала 

In [8]:
with open("Zabrodin_chronology.txt", "w", encoding="utf-8") as f:
    f.write(f"Хронологія дій персонажа: {CHARACTER}\n\n")
    f.write(chronology)